# The Asteroid's Point of View: every method compared (Kaggle)

This notebook runs our **training-free path rule** against **nine Earth's-view models** on JPL data:

- the three further models of the paper's Table IX: a **CNN**, **GPBoost** and a **tuned random forest**;
- the six models of prior work from Table IV: logistic regression, k-NN, random forest, XGBoost, LightGBM and an MLP.

The label is JPL's PHA flag for every method. The run reproduces Tables IV and IX, checks every fold against the
paper's stored results and runs exact paired McNemar tests. All the code is in `compare_models.py` in the attached
dataset.

**Before running:**

1. **Settings → Internet → On**, to install GPBoost.
2. **Accelerator: None (CPU).**
3. Optional: **Add Input → Your Work → Datasets → asteroid-model-comparison** (the uploaded zip). If no dataset is
   attached, the notebook downloads the same code and data from the project's public GitHub repository.

**Full run:** Save Version → Save & Run All (Commit). Allow 2–4 hours on Kaggle's CPU.
**Quick check:** set `QUICK = True` in the third code cell and click Run All (about 20–30 minutes).

In [ ]:
# Step 1. Install GPBoost, the only library Kaggle does not preinstall (needs Settings > Internet > On).
# The paper used gpboost 1.8.0; if that version is not available here, the latest one is installed.
!pip install -q gpboost==1.8.0 || pip install -q gpboost
!pip list 2>/dev/null | grep -i -E "^(gpboost|torch|scikit-learn|xgboost|lightgbm|numpy|pandas) "

In [ ]:
# Step 2. Get the files: from the attached dataset (the uploaded zip) or, if none is attached, from GitHub
import glob, os, subprocess, zipfile

REPO = "https://github.com/devkancheti4-design/asteroid-point-of-view.git"   # public repository of the paper

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)
                  + glob.glob(f"/kaggle/working/pkg/**/{name}", recursive=True)
                  + glob.glob(f"/kaggle/working/repo/**/{name}", recursive=True))
    return hits[0] if hits else None

SCRIPT = find("compare_models.py")
if SCRIPT is None:      # the uploaded zip is still zipped: unpack it into the working folder
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/working/pkg")
    SCRIPT = find("compare_models.py")
if SCRIPT is None:      # nothing attached (a fresh device): download the code and the data snapshot from GitHub
    if not os.path.isdir("/kaggle/working/repo/.git"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "/kaggle/working/repo"], check=True)
    for z in glob.glob("/kaggle/working/repo/datasets/neo_datasets_*.zip"):   # JPL data of 2026-11-11
        zipfile.ZipFile(z).extractall("/kaggle/working/repo/data/snapshot_tmp")
    SCRIPT = find("compare_models.py")
assert SCRIPT, "compare_models.py not found: attach the dataset, or turn Internet on (Settings) to download it"
for name in ["compare_models.py", "sbdb_neo.csv", "cad_earth.csv", "cad.csv", "sentry_risk_list.csv",
             "sheet_comparison_folds.csv", "earth_view_folds.csv"]:
    print(f"{name:28s} {find(name) or 'MISSING'}")

In [ ]:
# Step 3. Run every method. Each unit runs in its own process and saves its results as soon as it finishes.
# QUICK = True: only the first fold of each split (about 20-30 minutes), to check that everything works.
# QUICK = False: the paper's full comparison (allow 2-4 hours on Kaggle's CPU).
QUICK = False
UNITS = "B-cad B-sentry E-grouped E-row C-grouped C-row"   # all six units (see KAGGLE_GUIDE.md)
extra = "--max-folds 1" if QUICK else ""
!python "{SCRIPT}" --units {UNITS} {extra}

In [ ]:
# Step 4. The report and one table per unit (all files are in /kaggle/working/comparison_output/)
import pandas as pd
OUT = "/kaggle/working/comparison_output"
print(open(f"{OUT}/report.txt").read())
for unit in ["E-grouped", "C-grouped", "B-sentry", "B-cad", "E-row", "C-row"]:
    path = f"{OUT}/{unit}.csv"
    if os.path.exists(path):
        f = pd.read_csv(path)
        print(f"\n{unit} (mean over {f['fold'].nunique()} fold(s) or repeat(s))")
        display(f.groupby("method", sort=False)[["accuracy", "precision", "recall", "f1", "mcc", "roc_auc"]]
                .mean().round(4))

## Reading the results

- **The rule gives exactly the paper's numbers on any computer.** It has no training, and its geometry is
  deterministic.
- **The trained models can differ slightly from the paper on Kaggle's processors and library versions.** The paper's
  runs used macOS on Apple silicon with the versions in `requirements.txt`, and on that machine every number matches.
  The report's verdict states the largest difference.
- **Honest evaluations:** folds grouped by asteroid, and the two small NASA tables, where no object repeats. On
  these, the rule beats every model in every fold. Only the leaky row split lets GPBoost score higher, because its
  random effect per asteroid memorises asteroids it has already seen.
- **Downloads:** the **Output** tab of this notebook version holds `comparison_output/`. It contains the report, every
  fold's metrics, the row-level predictions and the paired tests.

The methods, their settings and their research papers are in `MODELS_AND_PAPERS.md` and `references.bib` in the
dataset.